In [1]:
import torch
from typing import Optional
from torch_geometric.utils import scatter

def broadcast(src: torch.Tensor, other: torch.Tensor, dim: int) -> torch.Tensor:
    """Broadcast `src` to the shape of `other` starting at dimension `dim`."""
    if dim < 0:
        dim = other.dim() + dim
    for _ in range(dim):
        src = src.unsqueeze(0)
    while src.dim() < other.dim():
        src = src.unsqueeze(-1)
    src = src.expand_as(other)
    return src


def scatter_softmax(
    src: torch.Tensor,
    index: torch.Tensor,
    dim: int = -1,
    dim_size: Optional[int] = None,
) -> torch.Tensor:
    """Scatter-aware softmax: compute softmax scores grouped by `index` along `dim`."""

    if not torch.is_floating_point(src):
        raise ValueError("`scatter_softmax` requires floating-point input tensors.")

    max_value_per_index = scatter(src, index, dim=dim, dim_size=dim_size, reduce="max")

    expanded_index = broadcast(index, src, dim)

    max_per_src_element = max_value_per_index.gather(dim, expanded_index)

    recentered_scores = src - max_per_src_element
    recentered_scores_exp = recentered_scores.exp()

    sum_per_index = scatter(
        recentered_scores_exp, index, dim=dim, dim_size=dim_size, reduce="sum"
    )
    normalizing_constants = sum_per_index.gather(dim, expanded_index)

    return recentered_scores_exp / normalizing_constants


class LinearReg(torch.nn.Module):
    """
    A linear regularization module that computes a scaling factor based on
    the number of atoms in a system, using learnable parameters.

    Attributes:
        param1 (torch.nn.Parameter):
            A learnable parameter affecting the scaling factor.
        param2 (torch.nn.Parameter):
            A learnable parameter controlling the influence of atom count.
        param3 (torch.nn.Parameter):
            A learnable parameter modifying the transformation of atom count.
        first_node (torch.Tensor):
            A buffer representing the first virtual node.
        n_min (torch.Tensor):
            The minimum number of atoms in the system.
        n_max (torch.Tensor):
            The maximum number of atoms in the system.
        self_loop (torch.Tensor):
            A buffer for self-loop regularization.

    Args:
        num_virt_nodes (int): Number of virtual nodes in the system.
        min_num_atoms (int): Minimum number of atoms in the dataset.
        max_num_atoms (int): Maximum number of atoms in the dataset.
    """

    def __init__(self, num_virt_nodes: int, min_num_atoms: int, max_num_atoms: int):
        super().__init__()
        self.param1 = torch.nn.Parameter(
            torch.normal(
                torch.ones(num_virt_nodes - 1) * 0.5,
                torch.ones(num_virt_nodes - 1) * 0.1,
            )
        )
        self.param2 = torch.nn.Parameter(torch.ones(num_virt_nodes - 1) * 4)
        self.param3 = torch.nn.Parameter(torch.ones(num_virt_nodes - 1) * 2)
        self.register_buffer("first_node", torch.ones(1))
        self.register_buffer("n_min", torch.tensor(min_num_atoms))
        self.register_buffer("n_max", torch.tensor(max_num_atoms))
        self.register_buffer("self_loop", torch.tensor(1.0))

    def forward(self, num_atoms: torch.Tensor):
        """
        Computes the regularization scaling factors for a batch of systems.

        Args:
            num_atoms (torch.Tensor): Tensor containing the number of atoms in each system.

        Returns:
            torch.Tensor: A tensor of regularization factors.
        """
        A = 1 + torch.abs(self.param2).unsqueeze(1)
        B = A * torch.tanh(torch.abs(self.param3)).unsqueeze(1)
        N = (num_atoms.unsqueeze(0) - self.n_min) / (self.n_max - self.n_min)
        alpha = torch.abs(A * (1 - N) + B * N)

        reg = torch.cat(
            (
                self.first_node.repeat(num_atoms.shape[0]).unsqueeze(0),
                torch.pow(torch.abs(self.param1.unsqueeze(1)), alpha),
            )
        )
        reg = reg.repeat_interleave(num_atoms, dim=1).flatten()

        reg = torch.cat([reg, self.self_loop.repeat(num_atoms.sum())])

        return reg

    def regularized_params(self):
        """
        Returns the parameters subject to regularization.

        Returns:
            torch.nn.Parameter: Regularized parameters.
        """
        return self.param1

e:\anaconda3\envs\tys\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:


class AggregationBlock(torch.nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        n_heads: int,
        basis_dim: int,
        activation: torch.nn.Module=torch.nn.SiLU(),
        **kwargs,
    ):
        super().__init__()

        if in_channels % n_heads != 0:
            raise ValueError(
                "The number of input attention channels must be divisible by the number of heads"
            )

        if out_channels % n_heads != 0:
            raise ValueError(
                "The number of output attention channels must be divisible by the number of heads"
            )

        self.n_heads = n_heads
        self.channels = out_channels
        self.hidden_channels = out_channels // self.n_heads

        self.lin_Q = torch.nn.Linear(out_channels, out_channels, bias=False)
        self.lin_K = torch.nn.Linear(in_channels, out_channels, bias=False)
        self.lin_V = torch.nn.Linear(in_channels, out_channels, bias=False)

        self.activation = torch.nn.LeakyReLU()

        self.attention = torch.nn.Parameter(
            torch.empty(1, n_heads, out_channels // n_heads)
        )

        self.basis_dim = basis_dim
        self.lin_E = torch.nn.Linear(basis_dim, out_channels, bias=False)

        self.output_layer = torch.nn.Sequential(
            torch.nn.LayerNorm(out_channels),
            activation,
        )

    def forward(
        self,
        senders: torch.Tensor,
        receivers: torch.Tensor,
        edge_indices: torch.Tensor,
        edge_attrs: torch.Tensor,
        *args,
    ) -> torch.Tensor:
        """
        Forward pass for the attention block.

        Args:
            senders (torch.Tensor): Feature matrix of sender nodes (N_senders x in_channels).
            receivers (torch.Tensor): Feature matrix of receiver nodes (N_receivers x out_channels).
            edge_indices (torch.Tensor): Edge index tensor (2, n_edges).
            edge_attrs (torch.Tensor): Edge feature tensor (n_edges x basis_dim).

        Returns:
            torch.Tensor: Updated node embeddings (N_receivers x out_channels).
        """
        E = self.lin_E(edge_attrs)

        Q = self.lin_Q(receivers)[edge_indices[1]]
        K = self.lin_K(senders)[edge_indices[0]]
        V = self.lin_V(senders)[edge_indices[0]]

        weights = torch.sum(
            self.attention
            * self.activation((Q + K + E).view(-1, self.n_heads, self.hidden_channels)),
            dim=2,
        )
        weights = scatter_softmax(weights, edge_indices[1], dim=0)
        weights = weights.unsqueeze(-1)

        V = V.view(-1, self.n_heads, self.hidden_channels)

        embedding = scatter(
            (weights * V).view(-1, self.channels),
            edge_indices[1],
            reduce="add",
            dim=0,
            dim_size=receivers.shape[0],
        )

        embedding = self.output_layer(embedding)

        return embedding

    def reset_parameters(self):
        """
        Reinitializes the model parameters.
        """
        init_xavier_uniform(self.lin_E)
        init_xavier_uniform(self.lin_Q)
        init_xavier_uniform(self.lin_K)
        init_xavier_uniform(self.lin_V)
        glorot(self.attention)


class BroadcastBlock(torch.nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        n_heads: int,
        basis_dim: int,
        activation: torch.nn.Module=torch.nn.SiLU(),
        **kwargs,
    ):
        super().__init__()

        if in_channels % n_heads != 0:
            raise ValueError(
                "The number of input attention channels must be divisible by the number of heads"
            )

        if out_channels % n_heads != 0:
            raise ValueError(
                "The number of output attention channels must be divisible by the number of heads"
            )

        self.n_heads = n_heads
        self.channels = in_channels
        self.hidden_channels = in_channels // self.n_heads

        self.lin_Q = torch.nn.Linear(out_channels, in_channels, bias=False)
        self.lin_K = torch.nn.Linear(out_channels, in_channels, bias=False)
        self.lin_V = torch.nn.Linear(out_channels, in_channels, bias=False)

        self.activation = torch.nn.LeakyReLU()

        self.attention = torch.nn.Parameter(
            torch.empty(1, n_heads, in_channels // n_heads)
        )

        self.basis_dim = basis_dim
        self.lin_E = torch.nn.Linear(basis_dim, in_channels, bias=False)

        self.weights_K = torch.nn.Parameter(
            torch.empty(n_heads, in_channels // n_heads, in_channels // n_heads)
        )
        self.weights_V = torch.nn.Parameter(
            torch.empty(n_heads, in_channels // n_heads, in_channels // n_heads)
        )

        self.output_layer = torch.nn.Sequential(
            torch.nn.Linear(in_channels, in_channels, bias=False),
            torch.nn.LayerNorm(in_channels),
            activation,
            torch.nn.Linear(in_channels, out_channels, bias=False),
        )

    def forward(
        self,
        senders: torch.Tensor,
        senders_self: torch.Tensor,
        receivers: torch.Tensor,
        edge_indices: torch.Tensor,
        edge_attrs: torch.Tensor,
        regularization_weights: torch.Tensor,
        *args,
    ) -> torch.Tensor:
        """
        Forward pass for the attention block.

        Args:
            senders (torch.Tensor): Feature matrix of sender nodes (N_senders x in_channels).
            senders_self (torch.Tensor): Feature matrix of sender nodes in self-loops (N_receivers x in_channels).
            receivers (torch.Tensor): Feature matrix of receiver nodes (N_receivers x in_channels).
            edge_indices (torch.Tensor): Edge index tensor (2, n_edges).
            edge_attrs (torch.Tensor): Edge feature tensor (n_edges x basis_dim).
            regularization_weights (torch.Tensor): .

        Returns:
            torch.Tensor: Updated node embeddings.
        """
        K = torch.vmap(torch.matmul, in_dims=(1, 0), out_dims=1)(
            senders.view(-1, self.n_heads, self.hidden_channels), self.weights_K
        ).reshape(-1, self.channels)

        K_self = self.lin_K(senders_self)
        K = torch.cat([K, K_self], dim=0)[edge_indices[0]]

        V = torch.vmap(torch.matmul, in_dims=(1, 0), out_dims=1)(
            senders.view(-1, self.n_heads, self.hidden_channels), self.weights_V
        ).reshape(-1, self.channels)
        V_self = self.lin_V(senders_self)
        V = torch.cat([V, V_self], dim=0)[edge_indices[0]]

        E = self.lin_E(edge_attrs)
        Q = self.lin_Q(receivers)[edge_indices[1]]

        weights = torch.sum(
            self.attention
            * self.activation((Q + K + E).view(-1, self.n_heads, self.hidden_channels)),
            dim=2,
        )
        weights = scatter_softmax(weights, edge_indices[1], dim=0)

        weights = weights.unsqueeze(-1)

        V = V.view(-1, self.n_heads, self.hidden_channels)

        embedding = scatter(
            regularization_weights.unsqueeze(1) * (weights * V).view(-1, self.channels),
            edge_indices[1],
            reduce="add",
            dim=0,
            dim_size=receivers.shape[0],
        )

        embedding = self.output_layer(embedding)

        return embedding

In [21]:
torch.arange(6) + 4

tensor([4, 5, 6, 7, 8, 9])

In [31]:
torch.cat([torch.rand(2,6), torch.rand(2,6)], dim=1)

tensor([[0.1722, 0.0828, 0.4969, 0.5428, 0.0952, 0.5424, 0.6122, 0.2937, 0.8814,
         0.8631, 0.1717, 0.6373],
        [0.5230, 0.3095, 0.3525, 0.1589, 0.9186, 0.8244, 0.0945, 0.3580, 0.0881,
         0.9964, 0.0719, 0.9602]])

In [3]:
agg = AggregationBlock(
    in_channels=6, # real_hidden_dim
    out_channels=4, # virt_hidden_dim (node)
    n_heads=2, # num_virt_heads
    basis_dim=7, # virt_basis_dim (edge)
)

real_node = torch.randn(8, 6) # 8个node，6个维度
virt_node = torch.randn(4, 4) # 4个node，4个维度
real2virt_edge = torch.randn(4*8, 7) # 40条edge，7个维度, 全连接+self_loop
real2virt_edgeindex = torch.cat([torch.randint(0,8,(4*8,1)), torch.randint(0,4,(4*8,1))], dim=1).T


agg(real_node, virt_node, real2virt_edgeindex, real2virt_edge)


tensor([[ 1.3211, -0.2772, -0.0687, -0.1165],
        [ 1.4670, -0.2295, -0.1867, -0.2015],
        [ 1.3957, -0.2653, -0.0702, -0.2024],
        [ 1.3683, -0.2733, -0.1559, -0.0819]], grad_fn=<SiluBackward0>)

In [14]:
broad = BroadcastBlock(
    in_channels=4, # virt_hidden_channels
    out_channels=6, # real_hidden_channels (node)
    n_heads=2, # num_virt_heads
    basis_dim=7, # virt_basis_dim (edge)
)
linreg = LinearReg(num_virt_nodes=4, min_num_atoms=4, max_num_atoms=1000)

virt2real_edgeindex = real2virt_edgeindex[(1,0),:] #torch.cat([real2virt_edgeindex[(1,0),:], torch.randint(0,8,(2,8))], dim=1)
virt2real_edge = torch.cat([real2virt_edge, torch.zeros(8,7)], dim=0)

reg_weights = linreg(torch.tensor([8]))
broad(virt_node, virt_node, real_node, virt2real_edgeindex, virt2real_edge, reg_weights)


RuntimeError: The size of tensor a (32) must match the size of tensor b (40) at non-singleton dimension 0